# PyTorch Tutorial - Emotion Detection in Images of Faces

Welcome to the first assignment of week 2. In this assignment, you will:
1. Learn to use PyTorch, a powerful deep learning framework.
2. See how you can in a couple of hours build a deep learning algorithm.

#### Why are we using PyTorch?

* PyTorch is one of the most popular deep learning frameworks alongside TensorFlow.
* It provides a flexible and intuitive interface for building neural networks.
* Being able to go from idea to result with the least possible delay is key to finding good models.
* PyTorch's dynamic computational graph makes debugging easier and experimentation faster.

## <font color='darkblue'>Updates</font>

#### Conversion to PyTorch
* This notebook has been converted from Keras/TensorFlow to PyTorch
* The model architecture and training approach remain conceptually the same
* PyTorch uses nn.Module class-based model definition instead of Keras Functional API
* Training loops are explicit instead of using model.fit()

## Load packages
* In this exercise, you'll work on the "Emotion detection" model.
* Let's load the required packages.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as transforms
from PIL import Image
from kt_utils import *
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow

%matplotlib inline

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

**Note**: In PyTorch, you define models as classes inheriting from `nn.Module`. Unlike Keras/TensorFlow, you don't need to create a separate session or graph. PyTorch uses dynamic computation graphs that are built on-the-fly during the forward pass.

## 1 - Emotion Tracking

* A nearby community health clinic is helping the local residents monitor their mental health.
* As part of their study, they are asking volunteers to record their emotions throughout the day.
* To help the participants more easily track their emotions, you are asked to create an app that will classify their emotions based on some pictures that the volunteers will take of their facial expressions.
* As a proof-of-concept, you first train your model to detect if someone's emotion is classified as "happy" or "not happy."

To build and train this model, you have gathered pictures of some volunteers in a nearby neighborhood. The dataset is labeled.
<img src="images/face_images.png" style="width:550px;height:250px;">

Run the following code to normalize the dataset and learn about its shapes.

In [ ]:
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_dataset()

# Normalize image vectors
X_train = X_train_orig / 255.
X_test = X_test_orig / 255.

# Reshape
Y_train = Y_train_orig.T
Y_test = Y_test_orig.T

print("number of training examples = " + str(X_train.shape[0]))
print("number of test examples = " + str(X_test.shape[0]))
print("X_train shape: " + str(X_train.shape))
print("Y_train shape: " + str(Y_train.shape))
print("X_test shape: " + str(X_test.shape))
print("Y_test shape: " + str(Y_test.shape))

**Details of the "Face" dataset**:
- Images are of shape (64,64,3)
- Training: 600 pictures
- Test: 150 pictures

## 2 - Building a model in PyTorch

PyTorch is very good for rapid prototyping. In just a short time you will be able to build a model that achieves outstanding results.

Here is an example of a model in PyTorch:

```python
class HappyModel(nn.Module):
    def __init__(self, input_shape):
        """
        input_shape: The height, width and channels as a tuple.
            Note that this does not include the 'batch' as a dimension.
            If you have a batch like 'X_train',
            then you can provide the input_shape using
            X_train.shape[1:]
        """
        super(HappyModel, self).__init__()
        
        # Define layers
        # Zero-Padding: pads the border with zeroes
        self.pad = nn.ZeroPad2d(3)
        
        # CONV -> BN -> RELU Block
        self.conv0 = nn.Conv2d(3, 32, kernel_size=7, stride=1)
        self.bn0 = nn.BatchNorm2d(32)
        
        # MAXPOOL
        self.max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # FLATTEN + FULLYCONNECTED
        # Calculate the flattened size after conv and pooling
        # You need to calculate this based on your architecture
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * 34 * 34, 1)  # Adjust dimensions accordingly
        
    def forward(self, x):
        # Forward pass through the network
        x = self.pad(x)
        x = self.conv0(x)
        x = self.bn0(x)
        x = torch.relu(x)
        x = self.max_pool(x)
        x = self.flatten(x)
        x = torch.sigmoid(self.fc(x))
        return x
```

#### PyTorch model structure

* In PyTorch, you define all layers in the `__init__` method
* The `forward` method defines how data flows through the network
* Unlike Keras, PyTorch doesn't reuse variable names - each layer is stored as an attribute

**Exercise**: Implement a `HappyModel` class.
* This assignment is more open-ended than most.
* Start by implementing a model using the architecture we suggest, and run through the rest of this assignment using that as your initial model.
* Later, come back and try out other model architectures.
* For example, you might take inspiration from the model above, but then vary the network architecture and hyperparameters however you wish.
* You can also use other functions such as `nn.AvgPool2d()`, `nn.AdaptiveAvgPool2d()`, `nn.Dropout()`.

**Note**: Be careful with your data's shapes. Use what you've learned in the videos to make sure your convolutional, pooling and fully-connected layers are adapted to the volumes you're applying it to.

In [ ]:
# GRADED FUNCTION: HappyModel

class HappyModel(nn.Module):
    """
    Implementation of the HappyModel.
    
    Arguments:
    input_shape -- shape of the images of the dataset
        (height, width, channels) as a tuple.
        Note that this does not include the 'batch' as a dimension.
        If you have a batch like 'X_train',
        then you can provide the input_shape using
        X_train.shape[1:]
    """
    def __init__(self, input_shape):
        super(HappyModel, self).__init__()
        
        ### START CODE HERE ###
        # Feel free to use the suggested outline in the text above to get started, and run through the whole
        # exercise (including the later portions of this notebook) once. Then come back also try out other
        # network architectures as well.
        
        
        ### END CODE HERE ###
    
    def forward(self, x):
        ### START CODE HERE ###
        
        
        ### END CODE HERE ###
        return x

You have now built a function to describe your model. To train and test this model, there are four steps in PyTorch:
1. Create the model by instantiating the class above

2. Define the loss function and optimizer:
   ```python
   criterion = nn.BCELoss()  # Binary Cross Entropy for binary classification
   optimizer = optim.Adam(model.parameters(), lr=0.001)
   ```

3. Train the model using a training loop:
   ```python
   for epoch in range(num_epochs):
       for inputs, labels in train_loader:
           optimizer.zero_grad()
           outputs = model(inputs)
           loss = criterion(outputs, labels)
           loss.backward()
           optimizer.step()
   ```

4. Test the model by evaluating on test data

If you want to know more about PyTorch training loops, refer to the official [PyTorch documentation](https://pytorch.org/tutorials/).

#### Step 1: create the model.
**Hint**:
The `input_shape` parameter is a tuple (height, width, channels). It excludes the batch number.
Try `X_train.shape[1:]` as the `input_shape`.

In [ ]:
### START CODE HERE ### (1 line)
happy_model = None
### END CODE HERE ###

# Move model to device
if happy_model is not None:
    happy_model = happy_model.to(device)
    print(happy_model)

#### Step 2: define loss and optimizer

**Hint**:
Optimizers you can try include `optim.Adam`, `optim.SGD` or others. See the documentation for [optimizers](https://pytorch.org/docs/stable/optim.html)
The "happiness detection" is a binary classification problem. The loss function that you can use is `nn.BCELoss()` (Binary Cross Entropy Loss).

In [ ]:
### START CODE HERE ### (2 lines)
criterion = None
optimizer = None
### END CODE HERE ###

#### Step 3: prepare data loaders

PyTorch uses DataLoader objects to handle batching and shuffling of data.

In [ ]:
# Convert numpy arrays to PyTorch tensors
# Note: PyTorch uses (N, C, H, W) format, while TensorFlow/Keras uses (N, H, W, C)
X_train_tensor = torch.FloatTensor(X_train).permute(0, 3, 1, 2)  # Convert to (N, C, H, W)
Y_train_tensor = torch.FloatTensor(Y_train)
X_test_tensor = torch.FloatTensor(X_test).permute(0, 3, 1, 2)
Y_test_tensor = torch.FloatTensor(Y_test)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

# Create data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

#### Step 4: train the model

**Hint**:
Use integers for the epochs.

**Note**: If you run the training cell again, the model will continue to train with the parameters it has already learned instead of reinitializing them.

In [ ]:
### START CODE HERE ###
num_epochs = 40  # You can adjust this

if happy_model is not None and criterion is not None and optimizer is not None:
    for epoch in range(num_epochs):
        happy_model.train()  # Set model to training mode
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = happy_model(inputs)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Statistics
            running_loss += loss.item() * inputs.size(0)
            predicted = (outputs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = correct / total
        
        if (epoch + 1) % 5 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}')
### END CODE HERE ###

#### Step 5: evaluate model
**Hint**:
Use the test data to evaluate the model's performance.

In [ ]:
### START CODE HERE ###
if happy_model is not None and criterion is not None:
    happy_model.eval()  # Set model to evaluation mode
    test_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # No gradients needed for evaluation
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = happy_model(inputs)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item() * inputs.size(0)
            predicted = (outputs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_loss = test_loss / len(test_dataset)
    test_accuracy = correct / total
    
    print()
    print("Loss = " + str(test_loss))
    print("Test Accuracy = " + str(test_accuracy))
### END CODE HERE ###

#### Expected performance
If your `HappyModel` function worked, its accuracy should be better than random guessing (50% accuracy).

To give you a point of comparison, our model gets around **95% test accuracy in 40 epochs** (and 99% train accuracy) with a mini batch size of 16 and "adam" optimizer.

#### Tips for improving your model

If you have not yet achieved a very good accuracy (>= 80%), here are some tips:

- Use blocks of CONV->BATCHNORM->RELU such as:
```python
self.conv0 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
self.bn0 = nn.BatchNorm2d(32)
# In forward:
x = self.conv0(x)
x = self.bn0(x)
x = torch.relu(x)
```
until your height and width dimensions are quite low and your number of channels quite large (≈32 for example).
You can then flatten the volume and use a fully-connected layer.
- Use MaxPool2d after such blocks. It will help you lower the dimension in height and width.
- Change your optimizer. We find Adam works well.
- If you get memory issues, lower your batch_size (e.g. 12)
- Run more epochs until you see the train accuracy no longer improves.

**Note**: If you perform hyperparameter tuning on your model, the test set actually becomes a dev set, and your model might end up overfitting to the test (dev) set. Normally, you'll want separate dev and test sets. The dev set is used for parameter tuning, and the test set is used once to estimate the model's performance in production.

## 3 - Conclusion

Congratulations, you have created a proof of concept for "happiness detection"!

## Key Points to remember
- PyTorch is a tool we recommend for rapid prototyping. It allows you to quickly try out different model architectures.
- Remember the four steps in PyTorch:

1. Create Model (define class inheriting from nn.Module)
2. Define Loss and Optimizer
3. Train (write explicit training loop)
4. Evaluate/Test

## 4 - Test with your own image (Optional)

Congratulations on finishing this assignment. You can now take a picture of your face and see if it can classify whether your expression is "happy" or "not happy". To do that:

1. Click on "File" in the upper bar of this notebook, then click "Open" to go on your Coursera Hub.
2. Add your image to this Jupyter Notebook's directory, in the "images" folder
3. Write your image's name in the following code
4. Run the code and check if the algorithm is right (0 is not happy, 1 is happy)!

The training/test sets were quite similar; for example, all the pictures were taken against the same background (since a front door camera is always mounted in the same position). This makes the problem easier, but a model trained on this data may or may not work on your own data. But feel free to give it a try!

In [ ]:
### START CODE HERE ###
img_path = 'images/my_image.jpg'
### END CODE HERE ###

# Load and preprocess the image
img = Image.open(img_path).resize((64, 64))
imshow(np.array(img))

# Convert to tensor
img_array = np.array(img) / 255.0
x = torch.FloatTensor(img_array).permute(2, 0, 1).unsqueeze(0)  # Add batch dimension
x = x.to(device)

# Make prediction
if happy_model is not None:
    happy_model.eval()
    with torch.no_grad():
        prediction = happy_model(x)
    print(f"\nPrediction: {prediction.item():.4f}")
    print(f"Result: {'Happy' if prediction.item() > 0.5 else 'Not Happy'}")

## 5 - Other useful functions in PyTorch (Optional)

Two other basic features of PyTorch that you'll find useful are:
- `print(model)`: prints the details of your model architecture
- `torchsummary` or `torchinfo`: libraries that provide detailed summaries similar to Keras

Run the following code.

In [ ]:
if happy_model is not None:
    print(happy_model)
    
    # Optional: if you have torchinfo installed
    # from torchinfo import summary
    # summary(happy_model, input_size=(1, 3, 64, 64))